# GPT-6.1 Terra Results — Task 1, 2a and 2b (Table S2)

Recomputes the GPT-6.1 Terra numbers of the S4 Appendix (Table S2) straight from the published
inputs, captions and results. Pull them from the Hub first (from the repository root):

```bash
mllmsent hub pull-datasets --groups inputs captions results
```

- **Task 1** scores Terra's direct answers (`data/gpt61-terra-only/`) against the
  `gpt4-openai-classify` labels with `mllmsent.evaluation.metrics.kfold_task1_eval`, the routine that
  reproduces Table S1.
- **Task 2a** (not fine-tuned) and **Task 2b** (fine-tuned) read each cell's `logs/test_logs.csv`
  for BART, ModernBERT and LLaMA-3 (qLoRA) on Terra's descriptions.

The paper reports `ci_t`, the Student-t 95% CI over the five folds ($t_{0.975,4} \times \mathrm{SEM}$,
as `compute_ci`); `ci_z` ($1.96 \times \mathrm{SEM}$) is shown beside it for reference.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
%matplotlib inline

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from mllmsent.evaluation.metrics import kfold_task1_eval

# ── Configuration ──────────────────────────────────────────────────────────────
DATA = ROOT / "data"
EXP = ROOT / "output" / "experiments"

SETTINGS = [("p5", 3), ("p3", 3), ("p5", 5), ("p3", 5)]
LLMS = (("BART", "bart-experiment"), ("MBERT", "modernbert-experiment"), ("LLAMA", "llama3-qlora"))
TASKS = (("2a", "experiments-not-finetuning"), ("2b", "experiments-finetuning"))

## Per-fold scores → mean ± 95% CI

In [ ]:
def summarise(f1):
    f = np.asarray(f1) * 100
    sem = stats.sem(f)
    return dict(f1=f.mean(), ci_z=1.96 * sem, ci_t=stats.t.ppf(0.975, len(f) - 1) * sem, folds=len(f))


rows = []
for p, s in SETTINGS:
    setting = f"<s{s},{p.upper()}>"
    gt = pd.read_csv(DATA / "gpt4-openai-classify" / f"percept_dataset_alpha{s}_{p}.csv")
    answers = pd.read_csv(DATA / "gpt61-terra-only" / f"alpha{s}{p}.csv")
    rows.append(dict(setting=setting, task="1", llm="-",
                     **summarise(kfold_task1_eval(gt, answers, "sentiment", 5)["f1_score"])))
    for task, tree in TASKS:
        for llm, token in LLMS:
            log = EXP / tree / f"terra-{token}-{p}-alpha{s}" / "logs" / "test_logs.csv"
            rows.append(dict(setting=setting, task=task, llm=llm,
                             **summarise(pd.read_csv(log)["f1_score"])))

table = pd.DataFrame(rows)
table.round({"f1": 1, "ci_z": 1, "ci_t": 1})

## Table S2 layout — F1 ± `ci_t` (%)

In [ ]:
order = [f"<s{s},{p.upper()}>" for p, s in SETTINGS]
rows_order = [("1", "-")] + [(task, llm) for task, _ in TASKS for llm, _ in LLMS]
cell = table.assign(value=table["f1"].map("{:.1f}".format) + " ± " + table["ci_t"].map("{:.1f}".format))
cell.pivot(index=["task", "llm"], columns="setting", values="value").loc[rows_order, order]

## F1 with 95% CI per setting

Task 1 is Terra answering directly; Task 2a / 2b are the text classifiers on Terra's descriptions,
before and after fine-tuning.

In [ ]:
COLORS = {"1": "#257de0", "2a": "#f47230", "2b": "#00b894"}
LABELS = {"1": "Task 1 (direct)", "2a": "Task 2a (not fine-tuned)", "2b": "Task 2b (fine-tuned)"}
X = {"-": 0, "BART": 1, "MBERT": 2, "LLAMA": 3}
OFFSET = {"1": 0.0, "2a": -0.14, "2b": 0.14}

fig, axes = plt.subplots(1, len(order), figsize=(16, 4.5), sharey=True)
for ax, setting in zip(axes, order):
    for task in ("1", "2a", "2b"):
        sub = table[(table["setting"] == setting) & (table["task"] == task)]
        x = sub["llm"].map(X) + OFFSET[task]
        ax.errorbar(x, sub["f1"], yerr=sub["ci_t"], fmt="o", ms=8, lw=2, capsize=4,
                    color=COLORS[task], label=LABELS[task])
    ax.set_title(setting)
    ax.set_xticks(list(X.values()), ["Terra", "BART", "ModernBERT", "LLaMA-3"])
    ax.set_ylim(0, 100)
    ax.grid(axis="y", color="#e5e5e5", lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Weighted F1 (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout()
plt.show()